In [ ]:
### Cell 1: Install Dependencies ###
!pip install -q optuna onnx onnxruntime


In [ ]:
### Cell 2: Downsample Dataset from Kaggle Input to Working Directory ###

import pathlib, random, shutil
import numpy as np
from PIL import Image
from collections import defaultdict

# ── Kaggle competition dataset paths
KAGGLE_INPUT_DIR = pathlib.Path('/kaggle/input/alaska2-image-steganalysis')
COVER_SRC = KAGGLE_INPUT_DIR / 'Cover'
STEGO_SRC = KAGGLE_INPUT_DIR / 'JUNIWARD'

# ── Writable working directory ──
WORKING_DIR = pathlib.Path('/kaggle/working/alaska2_dataset')

# check
assert COVER_SRC.is_dir(), f'Cover dir not found at {COVER_SRC}'
assert STEGO_SRC.is_dir(), f'JUNIWARD dir not found at {STEGO_SRC}'
print(f'✓ Input dataset located at {KAGGLE_INPUT_DIR}')

if WORKING_DIR.exists():
    print(f'Overwriting existing directory at {WORKING_DIR}...')
    shutil.rmtree(WORKING_DIR)

WORKING_DIR.mkdir(parents=True, exist_ok=True)
(WORKING_DIR / 'Cover').mkdir(exist_ok=True)
(WORKING_DIR / 'JUNIWARD').mkdir(exist_ok=True)

PAIRS_PER_QF = 15_000
TARGET_QFS = {75, 90, 95}
_BASE_LUM = np.array([
    16,11,10,16,24,40,51,61, 12,12,14,19,26,58,60,55,
    14,13,16,24,40,57,69,56, 14,17,22,29,51,87,80,62,
    18,22,37,56,68,109,103,77, 24,35,55,64,81,104,113,92,
    49,64,78,87,103,121,120,101, 72,92,95,98,112,100,103,99
], dtype=np.float64)

def _ijg_luminance_table(qf: int) -> np.ndarray:
    scale = 5000.0 / qf if qf < 50 else 200.0 - 2.0 * qf
    table = np.floor((_BASE_LUM * scale + 50.0) / 100.0)
    return np.clip(table, 1, 255).astype(np.uint8)

REF_TABLES = {qf: _ijg_luminance_table(qf) for qf in TARGET_QFS}

def detect_qf(jpeg_path: str) -> int | None:
    try:
        img = Image.open(jpeg_path)
        if not hasattr(img, 'quantization') or img.quantization is None: return None
        lum_qt = np.array(img.quantization[0], dtype=np.uint8)
        for qf, ref in REF_TABLES.items():
            if np.array_equal(lum_qt, ref): return qf
        return None
    except Exception: return None

print('Scanning JPEG quantization tables...')
qf_buckets = defaultdict(list)
cover_files = sorted(COVER_SRC.glob('*.jpg'))
print(f'  Total cover images found: {len(cover_files)}')

for i, cpath in enumerate(cover_files):
    qf = detect_qf(str(cpath))
    if qf is not None and qf in TARGET_QFS:
        if (STEGO_SRC / cpath.name).exists():
            qf_buckets[qf].append(cpath.name)
    if (i + 1) % 10_000 == 0:
        print(f'  scanned {i+1}/{len(cover_files)}')

for qf in sorted(TARGET_QFS):
    print(f'  QF {qf}: {len(qf_buckets[qf])} matched pairs')

random.seed(42)
selected_files = []
for qf in sorted(TARGET_QFS):
    pool = qf_buckets[qf]
    sampled = random.sample(pool, min(PAIRS_PER_QF, len(pool)))
    selected_files.extend(sampled)

print(f'Copying {len(selected_files)} downsampled pairs to {WORKING_DIR}...')
for fn in selected_files:
    shutil.copy(COVER_SRC / fn, WORKING_DIR / 'Cover' / fn)
    shutil.copy(STEGO_SRC / fn, WORKING_DIR / 'JUNIWARD' / fn)

print(f'✓ Downsampled dataset saved to {WORKING_DIR}.')


In [ ]:
### Cell 3: Read Dataset Pairs from Working Directory ###
import pathlib

WORKING_DIR = pathlib.Path('/kaggle/working/alaska2_dataset')
COVER_DIR = WORKING_DIR / 'Cover'
JUNIWARD_DIR = WORKING_DIR / 'JUNIWARD'

print('Reading available downsampled pairs...')
available_covers = sorted(COVER_DIR.glob('*.jpg'))
selected_pairs = [(cpath.name, 0) for cpath in available_covers]
print(f'✓ Found {len(selected_pairs)} pairs ready for training.')


In [ ]:
import torch
import numpy as np
import base64
import io

def load_srm_kernels() -> torch.Tensor:
    srm_b64 = "k05VTVBZAQBGAHsnZGVzY3InOiAnPGY0JywgJ2ZvcnRyYW5fb3JkZXInOiBGYWxzZSwgJ3NoYXBlJzogKDMwLCAxLCA1LCA1KSwgfSAgIAoAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAACAPwAAAAAAAAAAAAAAAAAAAAAAAIC/AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAIA/AAAAAAAAAAAAAAAAAACAvwAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAgL8AAIA/AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAIC/AAAAAAAAAAAAAAAAAAAAAAAAAAAAAIA/AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAACAvwAAAAAAAAAAAAAAAAAAAAAAAIA/AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAgL8AAAAAAAAAAAAAAAAAAIA/AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAgD8AAIC/AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAACAPwAAAAAAAAAAAAAAAAAAAAAAAAAAAACAvwAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAIA/AAAAAAAAAAAAAAAAAAAAAAAAAMAAAAAAAAAAAAAAAAAAAAAAAACAPwAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAgD8AAAAAAAAAAAAAAAAAAADAAAAAAAAAAAAAAAAAAACAPwAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAIA/AAAAwAAAgD8AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAgD8AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAMAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAgD8AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAgL8AAAAAAAAAAAAAAAAAAAAAAABAQAAAAAAAAAAAAAAAAAAAAAAAAEDAAAAAAAAAAAAAAAAAAAAAAAAAgD8AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAgL8AAAAAAAAAAAAAAAAAAEBAAAAAAAAAAAAAAAAAAABAwAAAAAAAAAAAAAAAAAAAgD8AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAACAPwAAQMAAAEBAAACAvwAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAIA/AAAAAAAAAAAAAAAAAAAAAAAAAAAAAEDAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAEBAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAIC/AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAgD8AAAAAAAAAAAAAAAAAAAAAAABAwAAAAAAAAAAAAAAAAAAAAAAAAEBAAAAAAAAAAAAAAAAAAAAAAAAAgL8AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAACAPwAAAAAAAAAAAAAAAAAAQMAAAAAAAAAAAAAAAAAAAEBAAAAAAAAAAAAAAAAAAACAvwAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAACAvwAAQEAAAEDAAACAPwAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAACAvwAAAAAAAAAAAAAAAAAAAAAAAAAAAABAQAAAAAAAAAAAAAAAAAAAAAAAAAAAAABAwAAAAAAAAAAAAAAAAAAAAAAAAAAAAACAPwAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAgL8AAABAAACAvwAAAAAAAAAAAAAAQAAAgMAAAABAAAAAAAAAAAAAAIC/AAAAQAAAgL8AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAIC/AAAAQAAAgL8AAAAAAAAAAAAAAEAAAIDAAAAAQAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAEAAAIC/AAAAAAAAAAAAAAAAAACAwAAAAEAAAAAAAAAAAAAAAAAAAABAAACAvwAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAQAAAgMAAAABAAAAAAAAAAAAAAIC/AAAAQAAAgL8AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAIC/AAAAQAAAAAAAAAAAAAAAAAAAAEAAAIDAAAAAAAAAAAAAAAAAAACAvwAAAEAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAACAvwAAAEAAAADAAAAAQAAAgL8AAABAAADAwAAAAEEAAMDAAAAAQAAAAMAAAABBAABAwQAAAEEAAADAAAAAQAAAwMAAAABBAADAwAAAAEAAAIC/AAAAQAAAAMAAAABAAACAvwAAgL8AAABAAAAAwAAAAEAAAIC/AAAAQAAAwMAAAABBAADAwAAAAEAAAADAAAAAQQAAQMEAAABBAAAAwAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAMAAAABAAACAvwAAAAAAAAAAAAAAQQAAwMAAAABAAAAAAAAAAAAAAEDBAAAAQQAAAMAAAAAAAAAAAAAAAEEAAMDAAAAAQAAAAAAAAAAAAAAAwAAAAEAAAIC/AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAMAAAABBAABAwQAAAEEAAADAAAAAQAAAwMAAAABBAADAwAAAAEAAAIC/AAAAQAAAAMAAAABAAACAvwAAgL8AAABAAAAAwAAAAAAAAAAAAAAAQAAAwMAAAABBAAAAAAAAAAAAAADAAAAAQQAAQMEAAAAAAAAAAAAAAEAAAMDAAAAAQQAAAAAAAAAAAACAvwAAAEAAAADAAAAAAAAAAAA="
    
    npy_data = base64.b64decode(srm_b64)
    kernels = np.load(io.BytesIO(npy_data))
    
    if kernels.shape == (5, 5, 1, 30):
        kernels = kernels.transpose(3, 2, 0, 1)
    elif kernels.shape == (30, 1, 5, 5):
        pass
    else:
        raise ValueError(f"Unexpected SRM kernel shape: {kernels.shape}")
    
    return torch.tensor(kernels, dtype=torch.float32)

SRM_KERNELS = load_srm_kernels()
print(f"✓ SRM kernels loaded: {SRM_KERNELS.shape}")



In [ ]:
### Cell 5: Yedroudj-Net Architecture
import torch.nn as nn
import torch.nn.functional as F

class SRMPreprocess(nn.Module):
    def __init__(self, srm_weights: torch.Tensor):
        super().__init__()
        self.register_buffer('weight', srm_weights)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out = F.conv2d(x, self.weight, bias=None, padding=2)
        return torch.abs(out)

class YedroudjNet(nn.Module):
    def __init__(self, srm_weights: torch.Tensor):
        super().__init__()
        self.srm = SRMPreprocess(srm_weights)
        self.conv1 = nn.Conv2d(30, 30, 5, padding=2)
        self.bn1   = nn.BatchNorm2d(30)
        self.conv2 = nn.Conv2d(30, 30, 3, padding=1)
        self.bn2   = nn.BatchNorm2d(30)
        self.conv3 = nn.Conv2d(30, 32, 3, padding=1)
        self.bn3   = nn.BatchNorm2d(32)
        self.conv4 = nn.Conv2d(32, 32, 3, padding=1)
        self.bn4   = nn.BatchNorm2d(32)
        self.conv5 = nn.Conv2d(32, 16, 3, padding=1)
        self.bn5   = nn.BatchNorm2d(16)
        self.fc1 = nn.Linear(16, 256)
        self.fc2 = nn.Linear(256, 1)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.srm(x)
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.avg_pool2d(x, kernel_size=5, stride=2, padding=2)
        x = F.relu(self.bn2(self.conv2(x)))
        x = F.avg_pool2d(x, kernel_size=3, stride=2, padding=1)
        x = F.relu(self.bn3(self.conv3(x)))
        x = F.avg_pool2d(x, kernel_size=3, stride=2, padding=1)
        x = F.relu(self.bn4(self.conv4(x)))
        x = F.avg_pool2d(x, kernel_size=3, stride=2, padding=1)
        x = F.relu(self.bn5(self.conv5(x)))
        x = F.adaptive_avg_pool2d(x, 1).flatten(1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x



In [ ]:
### Cell 6: Paired-Split Dataset & DataLoader ###
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from typing import List, Tuple

class PairedStegoDataset(Dataset):
    def __init__(self, pair_list: List[Tuple[str, int]], cover_dir: str, stego_dir: str):
        self.pairs = pair_list
        self.cover_dir = pathlib.Path(cover_dir)
        self.stego_dir = pathlib.Path(stego_dir)
        self.transform = transforms.Compose([
            transforms.Grayscale(num_output_channels=1),
            transforms.ToTensor(),
        ])

    def __len__(self) -> int:
        return len(self.pairs)

    def __getitem__(self, idx: int):
        fname, _ = self.pairs[idx]
        cover = Image.open(self.cover_dir / fname)
        stego = Image.open(self.stego_dir / fname)
        return self.transform(cover), self.transform(stego)

def paired_collate_fn(batch):
    covers, stegos = zip(*batch)
    covers = torch.stack(covers)
    stegos = torch.stack(stegos)
    images = torch.cat([covers, stegos], dim=0)
    labels = torch.cat([torch.zeros(len(covers), 1), torch.ones(len(stegos), 1)])
    return images, labels

def build_dataloaders(pair_list, cover_dir, stego_dir, half_batch=64, val_split=0.15, seed=42):
    rng = random.Random(seed)
    indices = list(range(len(pair_list)))
    rng.shuffle(indices)
    split = int(len(indices) * (1 - val_split))
    
    train_pairs = [pair_list[i] for i in indices[:split]]
    val_pairs   = [pair_list[i] for i in indices[split:]]

    train_ds = PairedStegoDataset(train_pairs, cover_dir, stego_dir)
    val_ds   = PairedStegoDataset(val_pairs,   cover_dir, stego_dir)

    train_loader = DataLoader(
        train_ds, batch_size=half_batch, shuffle=True,
        num_workers=4, pin_memory=True, collate_fn=paired_collate_fn, drop_last=True, prefetch_factor=2
    )
    val_loader = DataLoader(
        val_ds, batch_size=half_batch, shuffle=False,
        num_workers=4, pin_memory=True, collate_fn=paired_collate_fn, prefetch_factor=2
    )
    print(f"✓ Train pairs: {len(train_pairs)} | Val pairs: {len(val_pairs)}")
    print(f"  Target Half-Batch (N): {half_batch} | Effective Batch Size: {2 * half_batch}")
    print("  Using 4 workers and prefetch_factor=2 for stable throughput.")
    return train_loader, val_loader



In [ ]:
### Cell 7: Training Loop with Mixed Precision ###
from torch.amp import autocast, GradScaler
from tqdm.auto import tqdm

def train_one_epoch(model, loader, optimizer, criterion, device, scaler, epoch=None) -> float:
    model.train()
    running_loss, n_batches = 0.0, 0

    desc = f"Train Ep {epoch}" if epoch is not None else "Training"
    pbar = tqdm(loader, desc=desc, leave=False)

    for images, labels in pbar:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)

        with autocast(device_type='cuda'):
            logits = model(images)
            loss = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        loss_val = loss.item()
        running_loss += loss_val
        n_batches += 1
        pbar.set_postfix({'loss': f"{loss_val:.4f}"})

    return running_loss / max(n_batches, 1)

@torch.no_grad()
def evaluate(model, loader, criterion, device, epoch=None) -> dict:
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    tp, fp, tn, fn = 0, 0, 0, 0

    desc = f"Eval Ep {epoch}" if epoch is not None else "Evaluating"
    pbar = tqdm(loader, desc=desc, leave=False)

    for images, labels in pbar:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with autocast(device_type='cuda'):
            logits = model(images)
            loss = criterion(logits, labels)

        total_loss += loss.item()
        preds = (torch.sigmoid(logits) >= 0.5).float()
        correct += (preds == labels).sum().item()
        total += labels.numel()

        tp += ((preds == 1) & (labels == 1)).sum().item()
        fp += ((preds == 1) & (labels == 0)).sum().item()
        tn += ((preds == 0) & (labels == 0)).sum().item()
        fn += ((preds == 0) & (labels == 1)).sum().item()

    fpr = fp / max(fp + tn, 1)
    return {
        'val_loss': total_loss / max(len(loader), 1),
        'accuracy': correct / max(total, 1),
        'fpr': fpr, 'tp': tp, 'fp': fp, 'tn': tn, 'fn': fn,
    }



In [ ]:
### Cell 8: Optuna Hyperparameter Optimization ###
import optuna
from optuna.trial import Trial
import gc

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEVICE.type == 'cuda':
    torch.backends.cudnn.benchmark = True

N_TRIALS = 10
MAX_EPOCHS = 8
COVER_PENALTY_POS_WEIGHT = 0.6

# N=64 → effective batch 128. With T4x2 DataParallel, each GPU gets 64.
HALF_BATCH_N = 64 

def optuna_objective(trial: Trial) -> float:
    lr = trial.suggest_float('lr', 1e-5, 5e-3, log=True)
    weight_decay = trial.suggest_float('weight_decay', 1e-6, 1e-3, log=True)
    optimizer_name = trial.suggest_categorical('optimizer', ['Adam', 'AdamW'])

    model = YedroudjNet(SRM_KERNELS).to(DEVICE)
    if torch.cuda.device_count() > 1:
        print(f'Using {torch.cuda.device_count()} GPUs via DataParallel')
        model = nn.DataParallel(model)
    train_loader, val_loader = build_dataloaders(
        selected_pairs, str(COVER_DIR), str(JUNIWARD_DIR), half_batch=HALF_BATCH_N
    )

    OptClass = torch.optim.Adam if optimizer_name == 'Adam' else torch.optim.AdamW
    optimizer = OptClass(model.parameters(), lr=lr, weight_decay=weight_decay)

    pos_weight = torch.tensor([COVER_PENALTY_POS_WEIGHT], device=DEVICE)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    scaler = GradScaler()
    best_fpr = 1.0

    for epoch in range(MAX_EPOCHS):
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion, DEVICE, scaler, epoch=epoch+1)
        metrics = evaluate(model, val_loader, criterion, DEVICE, epoch=epoch+1)
        trial.report(metrics['fpr'], epoch)
        if trial.should_prune():
            raise optuna.TrialPruned()

        best_fpr = min(best_fpr, metrics['fpr'])
        print(f"  Trial {trial.number} | Ep {epoch+1}/{MAX_EPOCHS} | "
              f"val_loss={metrics['val_loss']:.4f} | FPR={metrics['fpr']:.4f}")

    del model, optimizer, scaler, train_loader, val_loader
    gc.collect()
    torch.cuda.empty_cache()
    return best_fpr

study = optuna.create_study(
    study_name='yedroudj-juniward-fpr', direction='minimize',
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=3),
)

study.optimize(optuna_objective, n_trials=N_TRIALS)



In [ ]:
### Cell 9: ONNX Export (YedroudjNet_JUNIWARD_Classification.onnx) ###
import onnx

ONNX_SAVE_PATH = '/kaggle/working/YedroudjNet_JUNIWARD_Classification.onnx'

def export_best_model_to_onnx(study: optuna.Study):
    best = study.best_trial.params
    print(f"Best hyperparameters: {best}")

    model = YedroudjNet(SRM_KERNELS).to(DEVICE)
    if torch.cuda.device_count() > 1:
        model = nn.DataParallel(model)
    train_loader, val_loader = build_dataloaders(
        selected_pairs, str(COVER_DIR), str(JUNIWARD_DIR), half_batch=HALF_BATCH_N
    )

    OptClass = torch.optim.Adam if best['optimizer'] == 'Adam' else torch.optim.AdamW
    optimizer = OptClass(model.parameters(), lr=best['lr'], weight_decay=best['weight_decay'])
    pos_weight = torch.tensor([COVER_PENALTY_POS_WEIGHT], device=DEVICE)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    scaler = GradScaler()

    FINAL_EPOCHS = 25
    for epoch in range(FINAL_EPOCHS):
        t_loss = train_one_epoch(model, train_loader, optimizer, criterion, DEVICE, scaler)
        metrics = evaluate(model, val_loader, criterion, DEVICE)
        print(f"  Final training | Epoch {epoch+1}/{FINAL_EPOCHS} | FPR={metrics['fpr']:.4f}")

    if isinstance(model, nn.DataParallel):
        model = model.module
    model.eval()
    model.to('cpu')
    dummy_input = torch.randn(1, 1, 512, 512)

    torch.onnx.export(
        model, dummy_input, ONNX_SAVE_PATH,
        export_params=True, opset_version=17, do_constant_folding=True,
        input_names=['input'], output_names=['logit'],
        dynamic_axes={'input': {0: 'batch_size'}, 'logit': {0: 'batch_size'}},
    )

    onnx_model = onnx.load(ONNX_SAVE_PATH)
    onnx.checker.check_model(onnx_model)
    print(f"\n✓ Exported strictly as {ONNX_SAVE_PATH}")

export_best_model_to_onnx(study)
